# Groww Stocks – Web Scraping Project
**Goal:** collect price, valuation and shareholding data for listed Indian stocks from groww.in, then clean, analyse and visualise it.

**Workflow:** A–Z list pages → stock URLs → each stock page → raw CSV → (next notebook) cleaning, EDA, dashboard, insights

**How to run:** run the cells top to bottom. Step A is a quick test on one stock. Scraping is polite (1–2 s delay) and saves a checkpoint file, so you can stop and re-run without losing data.

## 0. Imports & request headers

In [1]:
import os
import re
import time
import random
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

BASE = "https://groww.in"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.google.com/",
}

# slugs under /stocks/ that are NOT individual companies
NOT_STOCKS = {
    "stocks-list", "intraday", "filter", "calendar", "mtf", "screens", "sectors",
    "most-bought-stocks-on-groww", "stock-feed", "events",
}

## 1. Download a page (with retries)

In [2]:
def get_soup(url, retries=3, pause=2):
    """Download a page and return BeautifulSoup (None if it keeps failing)."""
    for attempt in range(retries):
        try:
            page = requests.get(url, headers=headers, timeout=20)
            if page.status_code == 200:
                return BeautifulSoup(page.text, "html.parser")
            print(f"  [{page.status_code}] {url}")
        except requests.RequestException as e:
            print(f"  [error] {url} -> {e}")
        time.sleep(pause * (attempt + 1))
    return None

## 2. Collect stock URLs from the A–Z list pages

In [3]:
STOCK_HREF = re.compile(r"^(?:https://groww\.in)?/stocks/([a-z0-9\-]+)/?$")


def stock_links_from_soup(soup):
    """Return {slug: company name} found on one list page."""
    found = {}
    for a in soup.find_all("a", href=True):
        m = STOCK_HREF.match(a["href"])
        if m and m.group(1) not in NOT_STOCKS:
            name = a.get("title") or a.get_text(strip=True)
            found[m.group(1)] = name
    return found


def collect_stock_urls(letters="abcdefghijklmnopqrstuvwxyz", delay=(1, 2)):
    """Walk every letter and every pagination page of /stocks/stocks-list/<letter>."""
    all_stocks = {}
    for letter in list(letters) + ["others"]:
        first = get_soup(f"{BASE}/stocks/stocks-list/{letter}")
        if first is None:
            continue
        # highest page number from the pagination links  .../a/2, .../a/3 ...
        pages = [1]
        for a in first.find_all("a", href=True):
            m = re.search(rf"/stocks/stocks-list/{letter}/(\d+)$", a["href"])
            if m:
                pages.append(int(m.group(1)))
        last_page = max(pages)

        all_stocks.update(stock_links_from_soup(first))
        for p in range(2, last_page + 1):
            time.sleep(random.uniform(*delay))
            soup = get_soup(f"{BASE}/stocks/stocks-list/{letter}/{p}")
            if soup is not None:
                all_stocks.update(stock_links_from_soup(soup))
        print(f"letter {letter.upper():>6}: {last_page} page(s), total stocks so far = {len(all_stocks)}")
        time.sleep(random.uniform(*delay))
    return all_stocks

## 3. Parse ONE stock page
Groww shows each metric as *label, then value* (e.g. `Market Cap` → `₹4,58,585Cr`), so we read the text line after each label and clean it with regex.

In [4]:
NUM = re.compile(r"^[-+]?₹?\s*[\d,]*\.?\d+\s*(?:Cr|%)?$")     # 1,147.95 | 33.24% | ₹4,58,585Cr
MISSING = re.compile(r"^(--|-|—|N/A|NA)$", re.I)


def page_lines(soup):
    """All visible text of the page as a clean list of lines."""
    text = soup.get_text("\n", strip=True).replace("’", "'")
    return [ln.strip() for ln in text.split("\n") if ln.strip()]


def to_number(s):
    """'₹4,58,585Cr' -> 458585.0   |   '33.24%' -> 33.24   |   None -> NaN"""
    if s is None:
        return np.nan
    s = re.sub(r"[₹,%\s]|Cr", "", s)
    try:
        return float(s)
    except ValueError:
        return np.nan


def value_after(lines, labels):
    """Value on the line right after a label line (label-value layout of Groww)."""
    for label in labels:
        for i in range(len(lines) - 1):
            if lines[i] == label:
                nxt = lines[i + 1]
                if NUM.match(nxt):
                    return nxt
                if MISSING.match(nxt):
                    return None
    return None


def regex_after(joined, labels):
    """Fallback: label and value stuck together in one piece of text."""
    for label in labels:
        m = re.search(re.escape(label) + r"\s*:?\s*(₹?\s*[-+]?[\d,]*\.?\d+\s*(?:Cr|%)?)", joined)
        if m:
            return m.group(1)
    return None


def number_field(lines, joined, labels):
    return to_number(value_after(lines, labels) or regex_after(joined, labels))


def text_after(lines, labels):
    """Text value after a label, e.g. Industry / CEO."""
    for label in labels:
        for i in range(len(lines) - 1):
            if lines[i] == label:
                return lines[i + 1]
    return np.nan


def parse_stock(soup, url):
    lines = page_lines(soup)
    joined = " ".join(lines)
    h1 = soup.find("h1")
    if h1 is None:
        return None                                    # not a normal stock page

    name = h1.get_text(strip=True)
    after_name = joined[joined.find(name):]
    m = re.search(r"₹\s*([\d,]+(?:\.\d+)?)", after_name)
    price = to_number(m.group(1)) if m else np.nan

    row = {
        "Company": name,
        "Symbol": text_after(lines, ["NSE symbol"]),
        "Industry": text_after(lines, ["Industry"]),
        "Price": price,
        "Prev_Close": number_field(lines, joined, ["Previous close", "Prev. close", "Prev close"]),
        "Open": number_field(lines, joined, ["Open price", "Open"]),
        "Day_Low": number_field(lines, joined, ["Today's low"]),
        "Day_High": number_field(lines, joined, ["Today's high"]),
        "Week52_Low": number_field(lines, joined, ["52 week low"]),
        "Week52_High": number_field(lines, joined, ["52 week high"]),
        "Volume": number_field(lines, joined, ["Live volume", "Volume"]),
        "Lower_Circuit": number_field(lines, joined, ["Lower circuit"]),
        "Upper_Circuit": number_field(lines, joined, ["Upper circuit"]),
        "Market_Cap_Cr": number_field(lines, joined, ["Market Cap"]),
        "ROE_pct": number_field(lines, joined, ["ROE"]),
        "PE_Ratio": number_field(lines, joined, ["P/E Ratio(TTM)", "P/E Ratio (TTM)"]),
        "EPS": number_field(lines, joined, ["EPS(TTM)", "EPS (TTM)"]),
        "PB_Ratio": number_field(lines, joined, ["P/B Ratio"]),
        "Dividend_Yield_pct": number_field(lines, joined, ["Dividend Yield"]),
        "Industry_PE": number_field(lines, joined, ["Industry P/E"]),
        "Book_Value": number_field(lines, joined, ["Book Value"]),
        "Debt_to_Equity": number_field(lines, joined, ["Debt to Equity"]),
        "Face_Value": number_field(lines, joined, ["Face Value"]),
        "CEO": text_after(lines, ["CEO/MD", "CEO", "Managing Director"]),
        "Founded_Year": number_field(lines, joined, ["Founded in"]),
    }

    # Shareholding pattern  ->  "DII 42.96%"  /  "DII42.96%"
    for cat in ["DII", "FII", "Public", "Promoters"]:
        m = re.search(rf"\b{cat}\s*([\d.]+)\s*%", joined)
        row[f"{cat}_pct"] = float(m.group(1)) if m else np.nan

    row["URL"] = url
    row["Scraped_At"] = datetime.now().strftime("%Y-%m-%d %H:%M")
    return row

## 4. Scrape many stocks (with checkpoint)

In [5]:
def scrape_stocks(slugs, out_csv="groww_raw_stocks.csv", delay=(1, 2), save_every=25):
    done = set()
    rows = []
    if os.path.exists(out_csv):                       # resume
        old = pd.read_csv(out_csv)
        rows = old.to_dict("records")
        done = set(old["URL"])
        print(f"Resuming: {len(done)} stocks already saved")

    for n, slug in enumerate(slugs, start=1):
        url = f"{BASE}/stocks/{slug}"
        if url in done:
            continue
        soup = get_soup(url)
        if soup is not None:
            row = parse_stock(soup, url)
            if row:
                rows.append(row)
        if n % save_every == 0:
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            print(f"{n}/{len(slugs)} done, {len(rows)} rows saved")
        time.sleep(random.uniform(*delay))

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    return df

---
# Run the scraper

### Step A – Test on one stock first
Check that every field is filled. If many are `NaN`, tell me which ones and I will fix the parser.

In [9]:
test_url = "https://groww.in/stocks/infosys-ltd"
soup = get_soup(test_url)
print("page downloaded:", soup is not None)
pd.Series(parse_stock(soup, test_url))

page downloaded: True


Company                                           Infosys
Symbol                                               INFY
Industry                                              NaN
Price                                              1165.3
Prev_Close                                         1168.0
Open                                               1167.0
Day_Low                                            1160.0
Day_High                                           1174.5
Week52_Low                                          980.4
Week52_High                                        1728.0
Volume                                          5293250.0
Lower_Circuit                                      1051.2
Upper_Circuit                                      1284.8
Market_Cap_Cr                                    473763.0
ROE_pct                                             33.24
PE_Ratio                                            15.62
EPS                                                 74.72
PB_Ratio      

### Step B – Collect all stock URLs (about 5 minutes, runs once)

In [10]:
if os.path.exists("groww_stock_urls.csv"):
    urls_df = pd.read_csv("groww_stock_urls.csv")
    all_stocks = dict(zip(urls_df["slug"], urls_df["name"]))
else:
    all_stocks = collect_stock_urls()
    pd.DataFrame({"slug": list(all_stocks), "name": list(all_stocks.values())}).to_csv("groww_stock_urls.csv", index=False)

print("Total stocks found:", len(all_stocks))

Total stocks found: 5987


### Step C – Pick a random sample
A random sample (instead of A…B…C in order) gives a mix of large and small companies. Change `N_STOCKS` for a bigger or smaller dataset. 800 stocks takes roughly 25–30 minutes.

In [12]:
N_STOCKS = 800
random.seed(42)
slugs = random.sample(list(all_stocks), min(N_STOCKS, len(all_stocks)))
len(slugs)

800

### Step D – Scrape (checkpoint file: `groww_raw_stocks.csv`)

In [13]:
df_raw = scrape_stocks(slugs, out_csv="groww_raw_stocks.csv")
df_raw.shape

25/800 done, 25 rows saved
50/800 done, 50 rows saved
75/800 done, 75 rows saved
100/800 done, 100 rows saved
125/800 done, 125 rows saved
150/800 done, 150 rows saved
175/800 done, 175 rows saved
200/800 done, 200 rows saved
225/800 done, 225 rows saved
250/800 done, 250 rows saved
275/800 done, 275 rows saved
300/800 done, 300 rows saved
325/800 done, 325 rows saved
350/800 done, 350 rows saved
375/800 done, 375 rows saved
400/800 done, 400 rows saved
425/800 done, 425 rows saved
450/800 done, 450 rows saved
475/800 done, 475 rows saved
500/800 done, 500 rows saved
525/800 done, 525 rows saved
550/800 done, 550 rows saved
575/800 done, 575 rows saved
600/800 done, 600 rows saved
625/800 done, 625 rows saved
650/800 done, 650 rows saved
675/800 done, 675 rows saved
700/800 done, 700 rows saved
725/800 done, 725 rows saved
750/800 done, 750 rows saved
775/800 done, 775 rows saved
800/800 done, 800 rows saved


(800, 31)

### Step E – Quick look at the raw data

In [17]:
df_raw.head()

,Company,Symbol,Industry,Price,Prev_Close,Open,Day_Low,Day_High,Week52_Low,Week52_High,...,Debt_to_Equity,Face_Value,CEO,Founded_Year,DII_pct,FII_pct,Public_pct,Promoters_pct,URL,Scraped_At
0,Tayo Rolls,NaN,NaN,89.40,89.40,87.41,87.41,91.94,52.00,52.00,...,-0.82,10.0,Mr. K. Shankar Marar,1968.0,0.01,NaN,26.78,73.21,https://groww.in/stocks/tayo-rolls-ltd,2026-10-22 10:43
1,Centenial Surgical Suture,NaN,NaN,71.25,71.25,71.25,71.25,71.25,71.25,124.90,...,0.38,10.0,Vijay Majrekar,1995.0,NaN,NaN,53.11,46.89,https://groww.in/stocks/centenial-surgical-sut...,2026-10-22 10:43
2,Afloat Enterprises,NaN,NaN,10.40,10.40,10.65,10.13,10.75,5.51,15.50,...,0.03,10.0,NaN,2015.0,NaN,NaN,83.37,16.63,https://groww.in/stocks/adishakti-loha-and-isp...,2026-10-22 10:43
3,Hybrid Financial Services,HYBRIDFIN,NaN,16.00,15.61,15.61,15.02,16.00,14.50,33.62,...,0.02,5.0,NaN,1986.0,0.37,NaN,35.61,64.02,https://groww.in/stocks/hybrid-financial-servi...,2026-10-22 10:43
4,Groww Nifty PSE ETF,NaN,NaN,91.25,90.81,90.81,89.73,91.35,89.30,109.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://groww.in/stocks/groww-nifty-pse-etf,2026-10-22 10:43


In [18]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Company             800 non-null    object 
 1   Symbol              340 non-null    object 
 2   Industry            0 non-null      float64
 3   Price               800 non-null    float64
 4   Prev_Close          714 non-null    float64
 5   Open                714 non-null    float64
 6   Day_Low             715 non-null    float64
 7   Day_High            715 non-null    float64
 8   Week52_Low          800 non-null    float64
 9   Week52_High         800 non-null    float64
 10  Volume              561 non-null    float64
 11  Lower_Circuit       739 non-null    float64
 12  Upper_Circuit       739 non-null    float64
 13  Market_Cap_Cr       700 non-null    float64
 14  ROE_pct             701 non-null    float64
 15  PE_Ratio            535 non-null    float64
 16  EPS     

In [19]:
df_raw.isnull().sum().sort_values(ascending=False)

Industry              800
FII_pct               476
Symbol                460
DII_pct               380
PE_Ratio              265
Volume                239
CEO                   170
Promoters_pct         123
Debt_to_Equity        103
Market_Cap_Cr         100
ROE_pct                99
Dividend_Yield_pct     98
Face_Value             98
Book_Value             98
Industry_PE            98
EPS                    98
PB_Ratio               98
Public_pct             95
Open                   86
Prev_Close             86
Day_Low                85
Day_High               85
Founded_Year           63
Upper_Circuit          61
Lower_Circuit          61
URL                     0
Company                 0
Week52_High             0
Week52_Low              0
Price                   0
Scraped_At              0
dtype: int64

In [22]:
import pandas as pd
df = pd.read_csv("groww_raw_stocks.csv")
bad = df[df["Symbol"].isnull()]["URL"].iloc[0]

for u in ["https://groww.in/stocks/infosys-ltd", bad]:
    lines = page_lines(get_soup(u))
    print(u)
    for i, l in enumerate(lines):
        if "Industry" in l or "NSE symbol" in l or "About" in l:
            print("  ", i, repr(l), "->", lines[i+1:i+3])

https://groww.in/stocks/infosys-ltd
   144 'Industry P/E' -> ['17.86', 'Book Value']
   166 'Industry P/E' -> ['The average P/E ratio of all the stocks in any sector. Different sectors consider different P/E ratios as healthy.', 'EPS(TTM)']
   216 'About' -> ["Infosys is a global leader in next-generation digital services and consulting, serving clients in over 59 countries with AI, cloud, and digital solutions. The company's purpose is to amplify human potential, creating new opportunities for people, businesses, and communities through technology. A core principle is to consistently surpass client expectations, a value encapsulated as 'Client value'. The company provides clients with an AI-first foundation, empowering their business with scalable enterprise AI and other digital technologies. Its always-on learning approach drives continuous improvement by building and transferring skills from its innovation ecosystem. During fiscal 2025, the company engaged with 1,869 active clients 

In [24]:
from industry_fix import test_one, add_industry
test_one("https://groww.in/stocks/infosys-ltd")

status: 200 | html length: 254073
INDUSTRY -> Information Technology | A: __NEXT_DATA__  props.pageProps.stockData.header.industryName

--- all industry/sector keys inside the hidden JSON ---
   props.pageProps.stockData.header.industryName = Information Technology
   props.pageProps.stockData.header.industryCodeV2 = GIC-00041


In [25]:
df = add_industry("groww_raw_stocks.csv")

Industry found for 798 of 800 stocks
Industry_Method
A: __NEXT_DATA__  props.pageProps.stockData.header.industryName    743
A: __NEXT_DATA__  props.pageProps.etfData.header.industryName       55
None                                                                 2
Name: count, dtype: int64


In [26]:
df["Industry"].value_counts().head(10)

Industry
Financial Services        99
Textiles                  54
Information Technology    45
Chemicals                 43
Real Estate               38
Engineering               36
Infrastructure            28
Pharmaceuticals           28
Food & Beverages          27
Distributors              26
Name: count, dtype: int64